# RISHI-Q GPU annotation (PD pilot)

Blinded ontology labeling with Qwen2.5-1.5B-Instruct on T4.
Prefer NA over YES. Unity ≠ entanglement.

**One LLM call per passage** (all features JSON) for tractable runtime.
Outputs: `/kaggle/working/annotations.parquet`, `manifest.json`

In [ ]:
# EMBEDDED_BUNDLE — blinded parquet + ontology (no private map)
import base64, zipfile, io
from pathlib import Path
_B64 = """..."""
EMBED = Path("/kaggle/working/_embedded_bundle")
EMBED.mkdir(parents=True, exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(_B64))).extractall(EMBED)
print("embedded", list(EMBED.iterdir()))


In [ ]:
print('=== /kaggle/input tree ===')
inp = Path('/kaggle/input')
if inp.exists():
    for p in sorted(inp.rglob('*')):
        if p.is_file() and p.suffix in {'.json','.yaml','.parquet','.bin','.safetensors'} or p.name in {'config.json','tokenizer.json'}:
            print(p)
        elif p.is_dir() and len(list(p.parts)) <= 6:
            print('DIR', p)
else:
    print('no /kaggle/input')
import os, sys, json, time, platform, subprocess, re, zipfile
from pathlib import Path
from datetime import datetime, timezone

OUTPUT = Path('/kaggle/working')
OUTPUT.mkdir(parents=True, exist_ok=True)

print(platform.python_version())

def run(cmd):
    try:
        return subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True, timeout=60)
    except Exception as e:
        return f'ERR:{type(e).__name__}:{e}'

diag = {
    'python': platform.python_version(),
    'platform': platform.platform(),
    'KAGGLE_KERNEL_RUN_TYPE': os.environ.get('KAGGLE_KERNEL_RUN_TYPE'),
    'NVIDIA_VISIBLE_DEVICES': os.environ.get('NVIDIA_VISIBLE_DEVICES'),
    'CUDA_VISIBLE_DEVICES': os.environ.get('CUDA_VISIBLE_DEVICES'),
    'ls_dev_nvidia': run(['bash','-lc','ls -la /dev/nvidia* 2>&1 || true']),
    'nvidia_smi': run(['bash','-lc','nvidia-smi 2>&1 || true']),
}
(OUTPUT / 'gpu_diag.json').write_text(json.dumps(diag, indent=2))
print(diag['nvidia_smi'][:500])

# Do NOT pip-install torch — replaces CUDA build with CPU wheels when GPU exists.
pkgs = []
for pkg in ['pyyaml', 'pandas', 'pyarrow', 'pydantic']:
    try:
        __import__(pkg if pkg != 'pyyaml' else 'yaml')
    except Exception:
        pkgs.append(pkg)
for mod, pkg in [('transformers','transformers'), ('accelerate','accelerate')]:
    try:
        __import__(mod)
    except Exception:
        pkgs.append(pkg)
if pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

import torch
print('torch', torch.__version__, 'cuda_built', torch.version.cuda)
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

ALLOW_CPU = os.environ.get('RISHIQ_ALLOW_CPU', '1') == '1'
if not torch.cuda.is_available():
    if not ALLOW_CPU:
        raise SystemExit('GPU not available and RISHIQ_ALLOW_CPU!=1')
    print('WARNING: no GPU attached — continuing on CPU (exploratory; slow)')
    DEVICE = 'cpu'
    # Prefer smaller model on CPU unless overridden
    os.environ.setdefault('RISHIQ_MODEL', 'Qwen/Qwen2.5-0.5B-Instruct')
    os.environ.setdefault('RISHIQ_MAX_PASSAGES', os.environ.get('RISHIQ_MAX_PASSAGES', '80'))
else:
    DEVICE = 'cuda'

# EMBEDDED prefer
EMBED = Path('/kaggle/working/_embedded_bundle')
inp_root = Path('/kaggle/input')
print('kaggle/input children:', list(inp_root.iterdir()) if inp_root.exists() else None)
INPUT = EMBED if (EMBED / 'blinded_passages.parquet').exists() else None
if INPUT is None and inp_root.exists():
    for p in sorted(inp_root.iterdir()):
        if (p / 'blinded_passages.parquet').exists():
            INPUT = p
            break
        # nested one level
        if p.is_dir():
            for q in p.iterdir():
                if q.name == 'blinded_passages.parquet':
                    INPUT = p
                    break
            if INPUT:
                break
if INPUT is None:
    # fallback: download attached dataset slug via API-less wget from Kaggle datasets download
    print('bundle mount missing — downloading dataset zip')
    dest = Path('/kaggle/working/bundle_dl')
    dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call([
        'kaggle', 'datasets', 'download', '-d', 'aks1321/rishiq-kaggle-bundle-public',
        '-p', str(dest), '--unzip',
    ])
    INPUT = dest if (dest / 'blinded_passages.parquet').exists() else next(
        p for p in dest.rglob('blinded_passages.parquet')
    ).parent

# also check new dataset mount layout
_extra = Path('/kaggle/input/datasets/aks1321/rishiq-kaggle-bundle-public')
if INPUT is None and _extra.exists() and (_extra/'blinded_passages.parquet').exists():
    INPUT = _extra
assert INPUT is not None and (INPUT / 'blinded_passages.parquet').exists(), f'bundle not found; input={list(inp_root.iterdir()) if inp_root.exists() else None}'
print('INPUT', INPUT, 'DEVICE', DEVICE)


In [ ]:
import pandas as pd
import yaml
from transformers import AutoModelForCausalLM, AutoTokenizer

blinded = pd.read_parquet(INPUT / 'blinded_passages.parquet')
ont = yaml.safe_load((INPUT / 'ontology_v0.1.yaml').read_text())
features = ont['features']
print('passages', len(blinded), 'features', len(features))

MAX_PASSAGES = int(os.environ.get('RISHIQ_MAX_PASSAGES', '40')) or len(blinded)
blinded = blinded.head(MAX_PASSAGES).reset_index(drop=True)
print('annotating', len(blinded))

# New Kaggle layout: /kaggle/input/models/<owner>/<model>/...
CANDIDATE_MODELS = [
    Path('/kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b-instruct/1'),
    Path('/kaggle/input/models/qwen-lm/qwen2.5/transformers/0.5b-instruct/1'),
    Path('/kaggle/input/qwen2.5/transformers/1.5b-instruct/1'),
    Path('/kaggle/input/qwen2.5/transformers/0.5b-instruct/1'),
]
# also discover any instruct dir under models
models_root = Path('/kaggle/input/models')
if models_root.exists():
    for p in models_root.rglob('config.json'):
        if 'instruct' in str(p).lower():
            CANDIDATE_MODELS.append(p.parent)

if DEVICE == 'cpu':
    prefer = [p for p in CANDIDATE_MODELS if '0.5b' in str(p)]
else:
    prefer = [p for p in CANDIDATE_MODELS if '1.5b' in str(p)]
local_model = next((p for p in prefer + CANDIDATE_MODELS if p.exists()), None)
if local_model is None:
    raise SystemExit(f'No local Qwen model found. Tried: {CANDIDATE_MODELS[:8]}')
MODEL = str(local_model)
print('MODEL', MODEL)
dtype = torch.float16 if DEVICE == 'cuda' else torch.float32
tokenizer = AutoTokenizer.from_pretrained(MODEL, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype, local_files_only=True)
model.to(DEVICE)
model.eval()
print('loaded', MODEL, 'on', DEVICE)


In [ ]:
feat_brief_all = [
    {
        'id': f['id'],
        'name': f['name'],
        'definition': f['definition'][:220],
        'exclusions': (f.get('exclusions') or '')[:160],
    }
    for f in features
]

SYSTEM = (
    'You label RISHI-Q structural ontology features on a passage. '
    'Prefer NA over 1 when unsure. Unity/oneness is NOT entanglement (Q06). '
    'Vibration is NOT QFT. Evidence spans must be exact substrings of the passage. '
    'Return ONLY a JSON list of objects: '
    '[{"feature_id":"O01","label":"1|0|NA|U","evidence":"...","reason":"...","confidence":0.0}, ...] '
    'Include EVERY requested feature id exactly once.'
)

BATCH = int(os.environ.get('RISHIQ_FEATURE_BATCH', '6'))
MAX_NEW = int(os.environ.get('RISHIQ_MAX_NEW_TOKENS', '900'))

def parse_list(raw: str):
    m = re.search(r'\[.*\]', raw, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def generate(prompt: str, max_new_tokens: int = MAX_NEW) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt')
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = out[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(gen, skip_special_tokens=True)

def annotate_passage(anonymous_id: str, text: str):
    by_id = {}
    raw_parts = []
    for start in range(0, len(feat_brief_all), BATCH):
        batch = feat_brief_all[start:start + BATCH]
        prompt = (
            f'Passage:\n{text[:1800]}\n\n'
            f'Features to label (JSON schemas):\n{json.dumps(batch)}\n\n'
            'Return JSON list covering every feature_id in this batch only.'
        )
        raw = generate(prompt)
        raw_parts.append(raw[:300])
        parsed = parse_list(raw)
        if isinstance(parsed, list):
            for item in parsed:
                if isinstance(item, dict) and item.get('feature_id'):
                    by_id[str(item['feature_id'])] = item
    rows = []
    for f in features:
        fid = f['id']
        item = by_id.get(fid) or {}
        label = str(item.get('label', 'NA')).upper().replace('YES', '1').replace('NO', '0')
        if label not in {'1', '0', 'NA', 'U'}:
            label = 'NA'
        evidence = str(item.get('evidence') or '')
        reason = str(item.get('reason') or 'missing_or_unparsed')
        conf = float(item.get('confidence') or 0.4)
        if label == '1':
            if not evidence or evidence.lower() not in text.lower():
                label = 'NA'
                reason += ';evidence_not_in_passage'
                evidence = ''
        rows.append({
            'passage_id': anonymous_id,
            'feature_id': fid,
            'label': label,
            'evidence': evidence if label == '1' else '',
            'reason': reason,
            'confidence': min(1.0, max(0.0, conf)),
            'annotator': 'transformers-annotator-batched',
            'model_version': f'{MODEL}@main',
            'prompt_version': 'ann-v0.3-batched6',
            'verified': False,
            'verification_flags': [],
        })
    return rows, ' | '.join(raw_parts)[:500]


In [ ]:
all_rows = []
t0 = time.time()
checkpoint = OUTPUT / 'annotations_partial.parquet'
for i, r in blinded.iterrows():
    aid = r['anonymous_id']
    text = r['text']
    try:
        rows, preview = annotate_passage(aid, text)
    except Exception as e:
        rows = [{
            'passage_id': aid,
            'feature_id': f['id'],
            'label': 'NA',
            'evidence': '',
            'reason': f'generation_error:{type(e).__name__}',
            'confidence': 0.1,
            'annotator': 'transformers-annotator-batched',
            'model_version': f'{MODEL}@main',
            'prompt_version': 'ann-v0.3-batched6',
            'verified': False,
            'verification_flags': ['error'],
        } for f in features]
        preview = str(e)
    all_rows.extend(rows)
    if (i + 1) % 5 == 0 or (i + 1) == len(blinded):
        pos = sum(1 for x in all_rows if x['label'] == '1')
        print(f'{i+1}/{len(blinded)} elapsed={time.time()-t0:.1f}s positives={pos}')
        pd.DataFrame(all_rows).to_parquet(checkpoint, index=False)

ann_df = pd.DataFrame(all_rows)
ann_path = OUTPUT / 'annotations.parquet'
ann_df.to_parquet(ann_path, index=False)
manifest = {
    'experiment_id': 'kaggle-annotation-pd-pilot-oneshot',
    'backend': 'transformers-annotator-batched',
    'model_name': MODEL,
    'prompt_version': 'ann-v0.3-batched6',
    'n_passages': len(blinded),
    'n_annotations': len(ann_df),
    'n_positive': int((ann_df['label'] == '1').sum()),
    'elapsed_sec': time.time() - t0,
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'gpu': DEVICE == 'cuda',
    'device': (torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu'),
    'allow_cpu_fallback': True,
    'note': 'EXPLORATORY — not confirmatory; blinded IDs require private map to join',
}
(OUTPUT / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print(manifest)
print('wrote', ann_path)
